In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import calendar
import holidays

In [7]:
#Load Data
df_daily= pd.read_csv('./data/D - Daily.csv', index_col=0)
df_interval = pd.read_csv('./data/D - Interval.csv', index_col=0)

#Prep Data
df_daily['Date'] = pd.to_datetime(df_daily['Date'].str[:8], format='%m/%d/%y')
df_daily = df_daily.sort_values('Date').reset_index(drop=True)
df_daily = df_daily.rename(columns={'Abandon Rate': 'Abandoned Rate'})
 
df_daily['day_of_week']  = df_daily['Date'].dt.dayofweek
df_daily['month']        = df_daily['Date'].dt.month
df_daily['day_of_month'] = df_daily['Date'].dt.day

for col in ['Call Volume', 'CCT', 'Abandoned Rate']:
    df_daily[col] = df_daily.groupby('day_of_week')[col].transform(
        lambda x: x.fillna(x.median())
    )

In [12]:
#prep inteveral data
start = '2025-04-01 00:00:00'
end = '2025-06-30 23:30:00'
dr = pd.date_range(start=start, end=end, freq='30min')
 
df_base = pd.DataFrame({
    'Month':           pd.Series(dr).dt.month_name(),
    'Day':             pd.Series(dr).dt.day,
    'Interval':        pd.Series(dr).dt.strftime('%H:%M:%S'),
    'Day of the Week': pd.Series(dr).dt.day_name()
})
 
df_interval = df_base.merge(right=df_interval, on=['Month', 'Day', 'Interval', 'Day of the Week'], how='left')
 
# Interpolate missing values, clip negatives
df_interval['Call Volume'] = df_interval['Call Volume'].interpolate(method='polynomial', order=2).clip(lower=0).astype(int)
df_interval['Service Level'] = df_interval['Service Level'].interpolate(method='polynomial', order=1)
df_interval['Abandoned Rate'] = df_interval['Abandoned Rate'].interpolate(method='linear').clip(lower=0)
df_interval['CCT'] = df_interval['CCT'].interpolate(method='polynomial', order=2).clip(lower=0)
 
mask = df_interval['Abandoned Calls'].isna()
df_interval.loc[mask, 'Abandoned Calls'] = (
    df_interval.loc[mask, 'Abandoned Rate'] * df_interval.loc[mask, 'Call Volume']
).astype(int)
 
df_interval['Portfolio'] = 'D'

In [13]:
# add time features
df_interval['Date']        = pd.to_datetime(df_interval['Month'] + ' ' + df_interval['Day'].astype(str) + ' 2025')
df_interval['day_of_week'] = df_interval['Date'].dt.dayofweek
df_interval['IntervalIdx'] = df_interval['Interval'].apply(
    lambda x: int(x.split(':')[0]) * 2 + int(x.split(':')[1]) // 30
)

In [17]:
# Merge daily totals into interval data
df_daily = df_daily.merge(
    df_daily[['Date', 'Call Volume']].rename(columns={'Call Volume': 'daily_cv'}),
    on='Date', how='left'
)
df_daily['cv_share'] = df_daily['Call Volume'] / df_daily['daily_cv']

KeyError: 'daily_cv'

In [ ]:
# compute intraday profiles
profile_cv  = df.groupby(['day_of_week', 'IntervalIdx'])['cv_share'].mean()
profile_cct = df.groupby(['day_of_week', 'IntervalIdx'])['CCT'].mean()
profile_abd = df.groupby(['day_of_week', 'IntervalIdx'])['Abandoned Rate'].mean()

KeyError: 'Column not found: cv_share'